# Chapter 2: Understanding and Preparing Video Data

## Introduction

Video is one of the most expressive and complex forms of data, capturing motion, color, and context across time. Its fusion of spatial and temporal information across multiple modalities makes it fundamental to many of today’s most advanced AI, Machine Learning (ML), and Computer Vision (CV) applications. From enabling autonomous vehicles to navigate urban environments to powering personalized recommendations, video data is central to how machines perceive and respond to the world.

Over the past several years, generative video tools have transformed from an aspirational research goal into accessible, production-ready technologies. This shift has unlocked powerful new possibilities, yet it also highlights the critical challenges of working with video. Its high dimensionality and multimodal nature make it difficult to store, analyze, and preprocess efficiently. Long-form video, in particular, demands careful organization to maintain temporal coherence and avoid overwhelming infrastructure.

Before we can conduct meaningful analysis or modeling, it is essential to build a structured environment for handling video datasets. This chapter will guide you step-by-step through the foundational tasks required to prepare high-quality video datasets for downstream AI workflows.

> **Why this matters:** Data pipelines for video quickly become complex as datasets scale. Establishing a clear structure now will save hours of rework and reduce errors later.

In this section, we’ll begin by setting up a dedicated workspace to organize your video datasets and prepare them for processing. This workspace will form the backbone of all subsequent tasks in the chapter.


In [ ]:
# Import necessary libraries
import os

# Initialize workspace with distinct directories for each data type
workspace = {
    'raw_data': './video_workshop/raw_data',
    'scenes': './video_workshop/scenes',
    'frames': './video_workshop/frames',
    'metadata': './video_workshop/metadata'
}

# Create directories if they don't exist
for path in workspace.values():
    os.makedirs(path, exist_ok=True)

print("Workspace directories created successfully!")

## Types of Video Data

Before we begin processing videos, it is essential to understand the different types of video data you may encounter. Each type carries unique characteristics and challenges that directly influence your data preparation strategy.

### Raw Video Data

Raw video consists of unprocessed footage captured directly from recording devices such as cameras, drones, or sensors. It retains the original resolution, frame rate, and encoding format, providing the highest level of detail and fidelity for analysis. This makes raw video an excellent source for applications where preserving every detail is critical.

> **Example:** Raw camera footage from autonomous vehicles is indispensable for tasks like object detection, depth estimation, and path planning.

However, raw video is also extremely large, making it difficult to store and transfer. High-resolution footage can quickly consume terabytes of space, creating a significant challenge when building scalable pipelines.

### Compressed Video Data

Compressed video uses codecs such as MPEG-4, H.264, or H.265 to reduce file size by eliminating redundancies between frames. This approach balances quality and storage efficiency, making it the standard for most video-sharing platforms and streaming services.

Compressed video is widely used for prototyping generative models and testing video workflows because of its smaller file size. However, compression introduces trade-offs, such as visual artifacts and reduced quality, that can hinder performance in applications requiring fine-grained detail (e.g., medical imaging).

### Annotated Video Data

Annotated video includes labels or metadata—such as bounding boxes, segmentation masks, or action tags—used to train supervised learning models. These enriched datasets are foundational for object detection, activity recognition, and caption generation.

While annotations add tremendous value, creating annotated datasets is often time-consuming and prone to human error. AI-assisted tools such as YOLO or Mask R-CNN can streamline the process, but scalability remains a challenge.

### Streaming Video Data

Streaming video is delivered and consumed in real-time, making it essential for latency-sensitive applications such as live surveillance, video conferencing, or sports analytics. Streaming data introduces unique challenges for AI workflows because it must be processed on-the-fly, often requiring edge computing solutions or GPU-accelerated inference.

### How This Chapter Handles Different Video Types

The examples in this chapter focus on **raw or compressed video files**, as these are the most common types used in AI pipelines. If you are working with annotated or streaming data, additional preprocessing may be necessary to align with the workflows we present.

> **Note:** Annotated and streaming video are not excluded from these methods; you will simply need to adapt the functions we build to account for additional metadata or real-time processing constraints.


## Characteristics of Video Data

Video data is unique among data types due to its combination of spatial, temporal, and often multimodal elements. Understanding these characteristics is crucial when designing your AI workflows.

### High Dimensionality

Each video frame represents a two-dimensional spatial structure embedded within a continuous temporal sequence. A single video file may contain thousands or even millions of frames, each carrying pixel-level detail. This level of complexity significantly increases the computational demands for both training and inference.

> **Example:** Detecting an action such as "throwing a ball" requires analyzing multiple frames in sequence. A single image would not capture the context needed to identify the action.

### Multimodal Nature

Video data often includes multiple modalities: visual content, audio, and sometimes textual elements like subtitles. These modalities work together to provide a more complete understanding of the video’s context.

> **Example:** Sentiment analysis of a video may rely on facial expressions (visual), tone of voice (audio), and spoken words (text) to infer emotional states.

### Common Challenges in Handling Video Data

Working with video data introduces several common obstacles:

1. **Storage and Bandwidth Constraints**: High-resolution formats like 4K and 8K require substantial storage space and bandwidth. Transferring or archiving raw video can be extremely costly.
2. **Complex Processing Requirements**: The spatial-temporal and multimodal nature of video demands sophisticated models that can process multiple signals in parallel.
3. **Annotation and Labeling**: Video annotation is time-intensive, requiring labels across frames and time sequences. Errors and inconsistencies are common.
4. **Scalability**: The volume, velocity, and variety of video data are growing rapidly, requiring scalable infrastructure for processing.
5. **Privacy and Ethical Concerns**: Video often contains personally identifiable information (PII), requiring strict compliance with regulations such as GDPR and CCPA.

> **Callout:** As your datasets scale, each of these challenges compounds. Proactively addressing them in your pipeline design will ensure smoother operations downstream.

**Next:** We will move into sourcing and collecting video datasets, bridging the gap between raw inputs and your organized workspace.


## Preparing Video Datasets – Sourcing and Collection

Once we understand the characteristics and challenges of video data, the next step is to gather and organize it effectively. This section will cover various strategies for sourcing and collecting video datasets, followed by practical examples of how to download and import videos into your workspace.


### Sourcing and Collecting Video Data

There are several common methods for acquiring video data:

1. **Publicly Available Datasets**: Benchmark datasets like UCF101, Kinetics-700, and AVA contain labeled video clips for activity recognition, object detection, and other tasks.
2. **Custom Video Capture**: Collect your own domain-specific data using cameras, drones, or sensors.
3. **Automated Filtering Pipelines**: Use automated tools to preprocess raw footage, such as segmenting into scenes or removing low-quality frames.
4. **Collaborative Data Sharing**: Partner with other organizations to access specialized datasets, such as medical or industrial video footage.

> **Callout:** Always ensure your datasets are ethically sourced and compliant with relevant regulations. This is especially critical when working with sensitive domains such as healthcare or surveillance.


### Downloading Sample Videos

To ensure that everyone can follow along, we’ll implement a helper function to download publicly available sample videos.


In [ ]:
import requests

def download_sample_videos(target_dir, count=3):
    """Download sample videos for the workshop

    Args:
        target_dir (str): Directory to save videos
        count (int): Number of sample videos to download

    Returns:
        list: Paths to downloaded videos
    """
    sample_urls = [
        "https://example.com/sample1.mp4",
        "https://example.com/sample2.mp4",
        "https://example.com/sample3.mp4",
        "https://example.com/sample4.mp4",
        "https://example.com/sample5.mp4"
    ]

    selected_urls = sample_urls[:min(count, len(sample_urls))]
    downloaded_paths = []

    for i, url in enumerate(selected_urls):
        filename = f"sample_video_{i+1}.mp4"
        filepath = os.path.join(target_dir, filename)

        if os.path.exists(filepath):
            print(f"File already exists: {filename}")
            downloaded_paths.append(filepath)
            continue

        try:
            print(f"Downloading: {filename}")
            response = requests.get(url, stream=True)
            with open(filepath, 'wb') as file:
                for chunk in response.iter_content(chunk_size=1024):
                    file.write(chunk)
            downloaded_paths.append(filepath)
            print(f"Downloaded: {filename}")
        except Exception as e:
            print(f"Error downloading {filename}: {e}")

    return downloaded_paths

# Example usage
# download_sample_videos(workspace['raw_data'], count=2)

### Importing Local Videos

In addition to downloading videos, you may want to import videos from a local directory:


In [ ]:
import shutil

def import_local_videos(source_dir, target_dir):
    """Import videos from a local directory

    Args:
        source_dir (str): Source directory containing videos
        target_dir (str): Target directory to copy videos

    Returns:
        list: Paths to imported videos
    """
    video_extensions = ('.mp4', '.avi', '.mov', '.mkv')
    video_files = []

    for root, _, files in os.walk(source_dir):
        for file in files:
            if file.lower().endswith(video_extensions):
                video_files.append(os.path.join(root, file))

    if not video_files:
        print(f"No video files found in {source_dir}")
        return []

    imported_paths = []
    for source_path in video_files:
        filename = os.path.basename(source_path)
        target_path = os.path.join(target_dir, filename)

        if os.path.exists(target_path):
            print(f"File already exists: {filename}")
            imported_paths.append(target_path)
            continue

        try:
            shutil.copy2(source_path, target_path)
            imported_paths.append(target_path)
            print(f"Imported: {filename}")
        except Exception as e:
            print(f"Error importing {filename}: {e}")

    return imported_paths

# Example usage
# import_local_videos('path/to/source/videos', workspace['raw_data'])


## Video Data Cleaning and Preprocessing Techniques

After acquiring video datasets, it is essential to clean and preprocess the data to ensure quality and consistency. Preprocessing allows you to remove redundant information, standardize formats, and optimize your dataset for downstream tasks.


### Key Preprocessing Steps

1. **Frame Extraction and Selection**: Reduce redundancy while preserving important temporal information. This can include adaptive frame selection or scene-based sampling.
2. **Resolution Standardization**: Ensure all videos have consistent resolution for uniform model input.
3. **Deduplication and Concept Balancing**: Remove duplicate content and ensure diversity in your dataset to avoid bias and overfitting.

> **Example:** Scene-based sampling can improve segmentation accuracy by focusing on keyframes near scene boundaries, while deduplication prevents over-representation of similar content.


### Analyzing Video Properties

Before applying cleaning techniques, it’s helpful to understand the properties of each video in your dataset. Let's create a function to extract metadata such as resolution, duration, and frame rate.


In [ ]:
import cv2

def analyze_video_properties(video_path):
    """Analyze and extract properties of a video file

    Args:
        video_path (str): Path to video file

    Returns:
        dict: Video properties
    """
    properties = {
        'path': video_path,
        'filename': os.path.basename(video_path),
        'filesize_mb': os.path.getsize(video_path) / (1024 * 1024)
    }

    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise ValueError(f"Could not open video: {video_path}")

        properties.update({
            'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
            'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
            'fps': cap.get(cv2.CAP_PROP_FPS),
            'frame_count': int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        })

        if properties['fps'] > 0 and properties['frame_count'] > 0:
            properties['duration_sec'] = properties['frame_count'] / properties['fps']
        else:
            properties['duration_sec'] = 0

        properties['resolution'] = f"{properties['width']}x{properties['height']}"
        cap.release()
    except Exception as e:
        properties['error'] = str(e)

    return properties

# Example usage:
# analyze_video_properties("path/to/video.mp4")


### Analyzing the Entire Dataset

We can now analyze all videos in our dataset and summarize their properties.


In [ ]:
import pandas as pd

def analyze_dataset(video_paths):
    """Analyze properties of a collection of videos

    Args:
        video_paths (list): List of video file paths

    Returns:
        pandas.DataFrame: Video properties
    """
    all_properties = []
    for video_path in video_paths:
        all_properties.append(analyze_video_properties(video_path))

    df = pd.DataFrame(all_properties)

    if not df.empty:
        print(f"Total videos: {len(df)}")
        if 'duration_sec' in df.columns:
            total_duration = df['duration_sec'].sum()
            avg_duration = df['duration_sec'].mean()
            print(f"Total duration: {total_duration:.1f} seconds ({total_duration/60:.1f} minutes)")
            print(f"Average duration: {avg_duration:.1f} seconds")

        if 'resolution' in df.columns:
            print("\nResolutions:")
            print(df['resolution'].value_counts())

        if 'fps' in df.columns:
            print("\nFrame rates:")
            print(df['fps'].value_counts())

    return df

# Example usage:
# dataset_info = analyze_dataset(list_of_video_paths)


### Preparing the Dataset for Processing

Finally, let's filter out problematic videos before downstream tasks such as scene detection.


In [ ]:
def prepare_dataset_for_processing(videos_df):
    """Prepare dataset for scene detection processing

    Args:
        videos_df (pandas.DataFrame): Videos dataset information

    Returns:
        list: List of video paths ready for processing
    """
    if videos_df.empty:
        print("No videos to prepare")
        return []

    valid_videos = videos_df[
        (videos_df['duration_sec'] > 0) &
        (videos_df['width'] > 0) & (videos_df['height'] > 0) &
        (videos_df['fps'] > 0) &
        ~videos_df.get('error', pd.Series(False))
    ]

    if len(valid_videos) < len(videos_df):
        print(f"Filtered out {len(videos_df) - len(valid_videos)} problematic videos")

    return valid_videos['path'].tolist()

# Example usage:
# ready_videos = prepare_dataset_for_processing(dataset_info)


## Tools and Libraries for Video Data Handling

Effectively handling video data requires a suite of specialized tools for preprocessing, scene segmentation, compression, augmentation, and format conversion. These tools form the backbone of any modern video workflow.


### Key Libraries for Video Processing

1. **OpenCV**  
   OpenCV is a widely used open-source computer vision library that supports real-time image and video processing. It enables tasks such as frame extraction, filtering, and edge detection.  
   - **Strengths:** Versatile, extensive documentation, and support for multiple programming languages.  
   - **Best For:** General-purpose computer vision tasks.

2. **FFmpeg**  
   FFmpeg is a high-performance command-line multimedia framework for encoding, decoding, transcoding, and streaming video and audio content.  
   - **Strengths:** Format conversion, compression, and batch augmentation.  
   - **Best For:** Large-scale dataset preprocessing and format standardization.

3. **PySceneDetect**  
   PySceneDetect is a Python-based library for detecting scene boundaries in videos by analyzing visual content changes between frames.  
   - **Strengths:** Automates video segmentation, integrates easily with Python workflows.  
   - **Best For:** Scene-based analysis and dataset preparation.

> **Tip:** Most robust pipelines use a combination of these tools to address different aspects of video data preparation.


### How These Tools Fit Into Our Workflow

The functions we implemented earlier in this chapter (for analyzing, cleaning, and preparing video data) already make use of **OpenCV** for extracting properties and filtering data. In later chapters, we will leverage **PySceneDetect** to segment videos into meaningful scenes.

**FFmpeg** can also be integrated into Python workflows using `subprocess` calls for tasks like re-encoding videos or extracting audio tracks.


In [ ]:
import subprocess

def convert_video_to_mp4(input_path, output_path):
    """Convert any video file to MP4 format using FFmpeg"""
    command = ["ffmpeg", "-i", input_path, "-c:v", "libx264", "-preset", "fast", "-crf", "22", output_path]
    try:
        subprocess.run(command, check=True)
        print(f"Converted {input_path} to {output_path}")
    except subprocess.CalledProcessError as e:
        print(f"Error during conversion: {e}")

# Example usage:
# convert_video_to_mp4("input.avi", "output.mp4")


## Hands-On Video Dataset Workshop

Now that we have covered the concepts, tools, and preprocessing techniques, let's put everything together in a hands-on workshop. This exercise will guide you through accessing, analyzing, organizing, and pre-processing video data using the functions we implemented in previous sections.


### Workshop Overview

In this workshop, you will:

1. Access and organize video datasets
2. Analyze their properties and visualize dataset characteristics
3. Clean and prepare the videos for downstream tasks such as scene detection

> **Goal:** By the end of this section, you will have a fully prepared video dataset ready for AI model training and analysis.


### Listing Available Videos

We will first list the videos that are currently available in our workspace.


In [ ]:
video_paths = []
for root, _, files in os.walk(workspace['raw_data']):
    for file in files:
        if file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
            video_paths.append(os.path.join(root, file))

print(f"Found {len(video_paths)} videos in the workspace")

if not video_paths:
    print("No videos found. Downloading sample videos...")
    video_paths = download_sample_videos(workspace['raw_data'], count=2)

### Analyzing the Dataset

Next, we will analyze the properties of each video to understand the dataset as a whole.


In [ ]:
dataset_info = analyze_dataset(video_paths)

### Visualizing Dataset Properties

We will create simple visualizations to help us better understand the dataset characteristics.


In [ ]:
import matplotlib.pyplot as plt

def visualize_dataset_properties(dataset_df):
    """Create visualizations of dataset properties"""
    if dataset_df.empty:
        print("No data to visualize")
        return

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    if 'duration_sec' in dataset_df.columns:
        axes[0, 0].hist(dataset_df['duration_sec'], bins=10, color='skyblue', edgecolor='black')
        axes[0, 0].set_title('Video Duration Distribution')
        axes[0, 0].set_xlabel('Duration (seconds)')
        axes[0, 0].set_ylabel('Count')

    if 'resolution' in dataset_df.columns:
        dataset_df['resolution'].value_counts().plot.bar(ax=axes[0, 1], color='lightgreen')
        axes[0, 1].set_title('Video Resolutions')
        axes[0, 1].set_xlabel('Resolution')
        axes[0, 1].set_ylabel('Count')

    if 'filesize_mb' in dataset_df.columns:
        axes[1, 0].hist(dataset_df['filesize_mb'], bins=10, color='salmon', edgecolor='black')
        axes[1, 0].set_title('File Size Distribution')
        axes[1, 0].set_xlabel('Size (MB)')
        axes[1, 0].set_ylabel('Count')

    if 'fps' in dataset_df.columns:
        dataset_df['fps'].value_counts().sort_index().plot.bar(ax=axes[1, 1], color='mediumpurple')
        axes[1, 1].set_title('Frame Rates')
        axes[1, 1].set_xlabel('FPS')
        axes[1, 1].set_ylabel('Count')

    plt.tight_layout()
    plt.show()

# Example usage:
visualize_dataset_properties(dataset_info)

### Preparing the Dataset for Scene Detection

Finally, we will filter out problematic videos and prepare the dataset for downstream tasks.


In [ ]:
ready_videos = prepare_dataset_for_processing(dataset_info)
print(f"{len(ready_videos)} videos ready for scene detection")

## Summary

In this chapter, we laid the groundwork for preparing video datasets for AI applications. We accomplished the following:

1. **Created a structured workspace** to organize raw data, scenes, frames, and metadata.
2. **Explored the types and characteristics of video data** and the unique challenges they present.
3. **Implemented methods for sourcing and collecting video datasets**, including downloading samples and importing local videos.
4. **Analyzed and cleaned the dataset**, extracting key properties and filtering out problematic videos.
5. **Leveraged powerful libraries** such as OpenCV, FFmpeg, and PySceneDetect to automate and optimize video data handling.
6. **Completed a hands-on workshop** to apply all of these techniques in a realistic, end-to-end workflow.

By the end of this process, you should have a clean and well-organized video dataset, ready for downstream tasks such as scene detection, annotation, and model training.


## Next Steps

In the next chapter, we will implement the **core scene detection algorithm** using PySceneDetect and other tools introduced here. You will learn how to split videos into meaningful segments that can be used for training models or performing analytics.


## Exercises

1. **Dataset Exploration**  
   Extend the `analyze_video_properties()` function to include additional metadata, such as audio channels, bit depth, or container format.

2. **Video Filtering**  
   Create a function to filter videos based on specific criteria such as:
   - Resolution (e.g., HD only)
   - Duration (e.g., under 3 minutes)
   - Frame rate thresholds

3. **Data Visualization**  
   Enhance the `visualize_dataset_properties()` function by adding a scatter plot showing the relationship between video duration and file size. This can help surface patterns that inform preprocessing choices.

> **Tip:** These exercises are designed to deepen your understanding of video data preparation. Consider applying them to a real-world dataset for maximum impact.
